<a href="https://colab.research.google.com/github/davesagit123/blank-app/blob/main/multi_bootstrapping_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import numpy as np
import pandas as pd

def detect_good_outliers_only(df, feature_cols, lower_is_better_cols=None, z_threshold=2.0, n_bootstraps=3000, random_state=42):
    """
    Identifies ONLY statistically superior ('good') outliers across horse ratings.
    Suppressing all negative/poor performance spikes.
    """
    np.random.seed(random_state)
    n = len(df)
    results = df[['Tab', 'Horse']].copy()

    if lower_is_better_cols is None:
        lower_is_better_cols = []

    good_outliers_summary = []
    z_matrix = np.zeros((n, len(feature_cols)))

    for c_idx, col in enumerate(feature_cols):
        raw_values = df[col].values.astype(float)
        is_low_better = col in lower_is_better_cols

        # Standardize direction so "Higher Z = Better Performance"
        val = -1.0 * raw_values if is_low_better else raw_values.copy()

        # Robust scale estimation
        med = np.median(val)
        mad = np.median(np.abs(val - med))
        scale = mad if mad > 0 else np.std(val) + 1e-6

        # Smooth Bootstrap Resampling
        boot_medians = np.zeros(n_bootstraps)
        boot_mads = np.zeros(n_bootstraps)

        for b in range(n_bootstraps):
            boot_idx = np.random.choice(n, size=n, replace=True)
            boot_sample = val[boot_idx] + np.random.normal(0, 0.05 * scale, size=n)
            boot_medians[b] = np.median(boot_sample)
            boot_mads[b] = np.median(np.abs(boot_sample - boot_medians[b])) + 1e-6

        pop_med = np.mean(boot_medians)
        pop_mad = np.mean(boot_mads)

        # Modified Z-Score against bootstrapped median
        mod_z = 0.6745 * (val - pop_med) / pop_mad
        z_matrix[:, c_idx] = mod_z

    # Extract Good Outliers Only (Z >= z_threshold)
    for i in range(n):
        horse_good_tags = []
        for c_idx, col in enumerate(feature_cols):
            z_val = z_matrix[i, c_idx]
            if z_val >= z_threshold:
                raw_val = df[col].iloc[i]
                horse_good_tags.append(f"{col}: {raw_val} (Z=+{z_val:.2f})")

        good_outliers_summary.append(", ".join(horse_good_tags) if horse_good_tags else "None")

    results['Good_Outlier_Flags'] = good_outliers_summary
    results['Max_Good_Z'] = np.round(np.max(np.maximum(z_matrix, 0), axis=1), 2)

    # Filter to display ONLY horses possessing at least one "Good Outlier" metric
    good_horses_df = results[results['Good_Outlier_Flags'] != "None"].reset_index(drop=True)
    return good_horses_df, results

## Example Usage of `detect_good_outliers_only`

First, let's create a DataFrame for the 'lower is better' data.

In [21]:
import io
import pandas as pd

lower_is_better_data = """
Tab	Horse	sum	Em	PP
1	CRYPTONIC	0.5	1.5	8.5
2	HEURISTIC	0.5	2	5.5
4	GRINZINGER POD	3.5	2	2.9
6	BETTER LATE	8	3.5	4.5
7	BROWN SUGAR	3.5	0	4
9	ANGARA	13.5	13.5	26
10	MISS CHECKONI	4	11.5	67
11	LIKA REMI	11.5	12	81
12	INCANTRESS	21	19.5	81
"""

df_lower_is_better = pd.read_csv(io.StringIO(lower_is_better_data), sep='\t')
display(df_lower_is_better)

,Tab,Horse,sum,Em,PP
0,1,CRYPTONIC,0.5,1.5,8.5
1,2,HEURISTIC,0.5,2.0,5.5
2,4,GRINZINGER POD,3.5,2.0,2.9
3,6,BETTER LATE,8.0,3.5,4.5
4,7,BROWN SUGAR,3.5,0.0,4.0
5,9,ANGARA,13.5,13.5,26.0
6,10,MISS CHECKONI,4.0,11.5,67.0
7,11,LIKA REMI,11.5,12.0,81.0
8,12,INCANTRESS,21.0,19.5,81.0


In [22]:
# Define feature columns and those where lower is better
feature_cols_lower = ['sum', 'Em', 'PP']
lower_is_better_cols = ['sum', 'Em', 'PP']

# Detect good outliers for 'lower is better' data
good_outliers_lower_df, all_results_lower_df = detect_good_outliers_only(
    df_lower_is_better,
    feature_cols=feature_cols_lower,
    lower_is_better_cols=lower_is_better_cols
)

print("\nGood Outliers (Lower is Better):")
display(good_outliers_lower_df)
print("\nAll Results (Lower is Better) with Outlier Flags:")
display(all_results_lower_df)


Good Outliers (Lower is Better):


,Tab,Horse,Good_Outlier_Flags,Max_Good_Z



All Results (Lower is Better) with Outlier Flags:


,Tab,Horse,Good_Outlier_Flags,Max_Good_Z
0,1,CRYPTONIC,None,1.03
1,2,HEURISTIC,None,1.14
2,4,GRINZINGER POD,None,1.36
3,6,BETTER LATE,None,1.23
4,7,BROWN SUGAR,None,1.30
5,9,ANGARA,None,0.00
6,10,MISS CHECKONI,None,0.34
7,11,LIKA REMI,None,0.00
8,12,INCANTRESS,None,0.00


Next, let's create a DataFrame for the 'higher is better' data.

In [23]:
import io
import pandas as pd

higher_is_better_data = """
Tab	Horse	CP	CF	TIM	JA	TA	JT	BP	D	$
1	CRYPTONIC	32	30	12	13	4	4	30	23	22
2	HEURISTIC	22	33	80	8	13	13	27	32	28
4	GRINZINGER POD	22	28	28	21	4	13	32	32	20
6	BETTER LATE	17	13	12	17	46	29	17	8	20
7	BROWN SUGAR	32	27	5	17	46	13	33	5	30
9	ANGARA	3	2	13	4	4	25	33	13	12
10	MISS CHECKONI	3	0	2	4	4	13	28	18	17
11	LIKA REMI	3	5	2	4	17	46	20	8	8
12	INCANTRESS	2	3	10	4	17	4	28	5	7
"""

df_higher_is_better = pd.read_csv(io.StringIO(higher_is_better_data), sep='\t')
display(df_higher_is_better)

,Tab,Horse,CP,CF,TIM,JA,TA,JT,BP,D,$
0,1,CRYPTONIC,32,30,12,13,4,4,30,23,22
1,2,HEURISTIC,22,33,80,8,13,13,27,32,28
2,4,GRINZINGER POD,22,28,28,21,4,13,32,32,20
3,6,BETTER LATE,17,13,12,17,46,29,17,8,20
4,7,BROWN SUGAR,32,27,5,17,46,13,33,5,30
5,9,ANGARA,3,2,13,4,4,25,33,13,12
6,10,MISS CHECKONI,3,0,2,4,4,13,28,18,17
7,11,LIKA REMI,3,5,2,4,17,46,20,8,8
8,12,INCANTRESS,2,3,10,4,17,4,28,5,7


In [24]:
# Define feature columns for 'higher is better' data (no 'lower is better' columns in this case)
feature_cols_higher = ['CP', 'CF', 'TIM', 'JA', 'TA', 'JT', 'BP', 'D', '$']

# Detect good outliers for 'higher is better' data
good_outliers_higher_df, all_results_higher_df = detect_good_outliers_only(
    df_higher_is_better,
    feature_cols=feature_cols_higher,
    lower_is_better_cols=[] # None of these features are 'lower is better'
)

print("\nGood Outliers (Higher is Better):")
display(good_outliers_higher_df)
print("\nAll Results (Higher is Better) with Outlier Flags:")
display(all_results_higher_df)


Good Outliers (Higher is Better):


,Tab,Horse,Good_Outlier_Flags,Max_Good_Z
0,2,HEURISTIC,"TIM: 80 (Z=+8.73), D: 32 (Z=+2.03)",8.73
1,4,GRINZINGER POD,"TIM: 28 (Z=+2.13), JA: 21 (Z=+2.75), D: 32 (Z=...",2.75
2,6,BETTER LATE,TA: 46 (Z=+3.90),3.90
3,7,BROWN SUGAR,TA: 46 (Z=+3.90),3.90
4,11,LIKA REMI,JT: 46 (Z=+3.45),3.45



All Results (Higher is Better) with Outlier Flags:


,Tab,Horse,Good_Outlier_Flags,Max_Good_Z
0,1,CRYPTONIC,None,1.83
1,2,HEURISTIC,"TIM: 80 (Z=+8.73), D: 32 (Z=+2.03)",8.73
2,4,GRINZINGER POD,"TIM: 28 (Z=+2.13), JA: 21 (Z=+2.75), D: 32 (Z=...",2.75
3,6,BETTER LATE,TA: 46 (Z=+3.90),3.90
4,7,BROWN SUGAR,TA: 46 (Z=+3.90),3.90
5,9,ANGARA,None,1.14
6,10,MISS CHECKONI,None,0.45
7,11,LIKA REMI,JT: 46 (Z=+3.45),3.45
8,12,INCANTRESS,None,0.54


## Applying `detect_good_outliers_only` (Version 2) to 'Lower is Better' Data

In [28]:
# Define feature columns and the directional map for 'lower is better'
feature_cols_lower_v2 = ['sum', 'Em', 'PP']
directional_map_lower_v2 = {'sum': 'lower', 'Em': 'lower', 'PP': 'lower'}

# Detect good outliers using Version 2 of the function
results_lower_v2 = detect_good_outliers_only(
    df_lower_is_better,
    feature_cols=feature_cols_lower_v2,
    directional_map=directional_map_lower_v2
)

print("\nResults for Lower is Better Data (Version 2):")
display(results_lower_v2[results_lower_v2['Is_Good_Outlier']])


Results for Lower is Better Data (Version 2):


,Tab,Horse,sum,Em,PP,Good_Outlier_Metrics,Max_Good_Z,Is_Good_Outlier


## Applying `detect_good_outliers_only` (Version 2) to 'Higher is Better' Data

In [29]:
# Define feature columns for 'higher is better'
feature_cols_higher_v2 = ['CP', 'CF', 'TIM', 'JA', 'TA', 'JT', 'BP', 'D', '$']

# Detect good outliers using Version 2 of the function
# For 'higher is better', we don't need to specify 'lower' in directional_map,
# so we can omit it or pass an empty dictionary if all are 'higher'.
# The function defaults to 'higher' if not specified for a column.
results_higher_v2 = detect_good_outliers_only(
    df_higher_is_better,
    feature_cols=feature_cols_higher_v2
)

print("\nResults for Higher is Better Data (Version 2):")
display(results_higher_v2[results_higher_v2['Is_Good_Outlier']])


Results for Higher is Better Data (Version 2):


,Tab,Horse,CP,CF,TIM,JA,TA,JT,BP,D,$,Good_Outlier_Metrics,Max_Good_Z,Is_Good_Outlier
1,2,HEURISTIC,22,33,80,8,13,13,27,32,28,"[TIM, D, $]",14.54,True
2,4,GRINZINGER POD,22,28,28,21,4,13,32,32,20,"[TIM, JA, D]",3.57,True
3,6,BETTER LATE,17,13,12,17,46,29,17,8,20,"[TA, JT]",4.46,True
4,7,BROWN SUGAR,32,27,5,17,46,13,33,5,30,"[TA, $]",4.46,True
7,11,LIKA REMI,3,5,2,4,17,46,20,8,8,[JT],6.22,True


version 2

In [30]:
def detect_good_outliers_only(df, feature_cols, directional_map=None, n_bootstraps=3000, z_threshold=2.25):
    import numpy as np
    results = df.copy()
    n = len(df)

    # Matrix to store Z-scores for each horse/feature
    scores_matrix = np.zeros((n, len(feature_cols)))

    for c_idx, col in enumerate(feature_cols):
        values = df[col].values.astype(float)

        # Invert if lower-is-better so 'superior' is always a positive Z-score
        if directional_map and directional_map.get(col) == 'lower':
            values = -1.0 * values

        # Robust scale estimation (MAD)
        med = np.median(values)
        mad = np.median(np.abs(values - med)) + 1e-6

        # Bootstrap to find the population baseline
        boot_meds = []
        for _ in range(n_bootstraps):
            boot_sample = np.random.choice(values, size=n, replace=True)
            # Add slight jitter to prevent ties in small samples
            boot_sample += np.random.normal(0, 0.05 * mad, size=n)
            boot_meds.append(np.median(boot_sample))

        pop_med = np.mean(boot_meds)
        pop_mad = np.std(boot_meds) + 1e-6 # Use bootstrap spread as the denominator

        # Calculate Modified Z-scores
        scores_matrix[:, c_idx] = (values - pop_med) / pop_mad

    # Filter: Only keep scores above threshold (the "Good" outliers)
    results['Good_Outlier_Metrics'] = [
        [feature_cols[j] for j in range(len(feature_cols)) if scores_matrix[i, j] > z_threshold]
        for i in range(n)
    ]
    results['Max_Good_Z'] = np.round(np.max(scores_matrix, axis=1), 2)
    results['Is_Good_Outlier'] = results['Good_Outlier_Metrics'].apply(lambda x: len(x) > 0)

    return results

sectionals method

## Outlier Detection for Time Data: 'LAST 3 RUNS'

In [31]:
import io
import pandas as pd

last_3_runs_data = """
Runner Name	Runner Time	Early Pace	L800m	L600m	L400m	L200m
1. Cryptonic	-2.87L	44	-1.20L	-1.07L	-0.78L	-0.04L
2. Heuristic	+21.93L	29	+2.80L	+2.86L	+3.24L	+3.19L
4. Grinzinger Pod	+0.89L	28	-1.24L	-1.54L	-1.33L	-0.81L
6. Better Late	+1.57L	37	-0.56L	-0.74L	-0.62L	-0.17L
7. Brown Sugar	-0.41L	58	-0.21L	-0.13L	-0.35L	-0.08L
9. Angara	+12.66L	37	+0.86L	+0.78L	+0.81L	+0.83L
10. Miss Checkoni	+2.55L	39	-0.58L	-0.39L	-0.04L	+0.35L
11. Lika Remi	-1.27L	42	-1.44L	-0.84L	+0.09L	-0.05L
12. Incantress	+3.20L	49	-0.37L	-0.01L	+0.38L	+0.54L
"""

df_last_3_runs = pd.read_csv(io.StringIO(last_3_runs_data), sep='\t')

# Clean up 'Runner Name' to extract just the number and rename it 'Tab'
df_last_3_runs['Tab'] = df_last_3_runs['Runner Name'].str.extract('^(\d+)\.').astype(int)
df_last_3_runs['Horse'] = df_last_3_runs['Runner Name'].str.replace('^\d+\. ', '', regex=True)
df_last_3_runs = df_last_3_runs.drop(columns=['Runner Name'])

# Convert 'L' (lengths) to numeric, stripping the 'L' and handling signs
for col in ['Runner Time', 'L800m', 'L600m', 'L400m', 'L200m']:
    df_last_3_runs[col] = df_last_3_runs[col].str.replace('L', '', regex=False).astype(float)

display(df_last_3_runs)

<>:20: SyntaxWarning: invalid escape sequence '\d'
<>:21: SyntaxWarning: invalid escape sequence '\d'
<>:20: SyntaxWarning: invalid escape sequence '\d'
<>:21: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_1127/2660342500.py:20: SyntaxWarning: invalid escape sequence '\d'
  df_last_3_runs['Tab'] = df_last_3_runs['Runner Name'].str.extract('^(\d+)\.').astype(int)
/tmp/ipykernel_1127/2660342500.py:21: SyntaxWarning: invalid escape sequence '\d'
  df_last_3_runs['Horse'] = df_last_3_runs['Runner Name'].str.replace('^\d+\. ', '', regex=True)


,Runner Time,Early Pace,L800m,L600m,L400m,L200m,Tab,Horse
0,-2.87,44,-1.20,-1.07,-0.78,-0.04,1,Cryptonic
1,21.93,29,2.80,2.86,3.24,3.19,2,Heuristic
2,0.89,28,-1.24,-1.54,-1.33,-0.81,4,Grinzinger Pod
3,1.57,37,-0.56,-0.74,-0.62,-0.17,6,Better Late
4,-0.41,58,-0.21,-0.13,-0.35,-0.08,7,Brown Sugar
5,12.66,37,0.86,0.78,0.81,0.83,9,Angara
6,2.55,39,-0.58,-0.39,-0.04,0.35,10,Miss Checkoni
7,-1.27,42,-1.44,-0.84,0.09,-0.05,11,Lika Remi
8,3.20,49,-0.37,-0.01,0.38,0.54,12,Incantress


In [32]:
# Define feature columns for 'LAST 3 RUNS' data
feature_cols_last_3 = ['Runner Time', 'Early Pace', 'L800m', 'L600m', 'L400m', 'L200m']

# Define directional map: 'Early Pace' is higher is better, all others are lower is better
directional_map_last_3 = {
    'Runner Time': 'lower',
    'Early Pace': 'higher',
    'L800m': 'lower',
    'L600m': 'lower',
    'L400m': 'lower',
    'L200m': 'lower'
}

# Detect good outliers using Version 2 of the function
results_last_3_runs = detect_good_outliers_only(
    df_last_3_runs,
    feature_cols=feature_cols_last_3,
    directional_map=directional_map_last_3
)

print("\nGood Outliers for LAST 3 RUNS:")
display(results_last_3_runs[results_last_3_runs['Is_Good_Outlier']])


Good Outliers for LAST 3 RUNS:


,Runner Time,Early Pace,L800m,L600m,L400m,L200m,Tab,Horse,Good_Outlier_Metrics,Max_Good_Z,Is_Good_Outlier
2,0.89,28,-1.24,-1.54,-1.33,-0.81,4,Grinzinger Pod,"[L600m, L400m, L200m]",3.49,True
4,-0.41,58,-0.21,-0.13,-0.35,-0.08,7,Brown Sugar,[Early Pace],5.55,True
7,-1.27,42,-1.44,-0.84,0.09,-0.05,11,Lika Remi,[L800m],2.36,True
8,3.20,49,-0.37,-0.01,0.38,0.54,12,Incantress,[Early Pace],2.83,True


## Outlier Detection for Time Data: 'LAST 10 RUNS'

In [33]:
import io
import pandas as pd

last_10_runs_data = """
Runner Name	Runner Time	Early Pace	L800m	L600m	L400m	L200m
1. Cryptonic	-1.69L	40	-1.22L	-1.16L	-0.77L	-0.19L
2. Heuristic	-0.51L	48	-0.68L	-0.49L	-0.18L	+0.33L
4. Grinzinger Pod	+2.22L	31	-1.18L	-1.16L	-0.93L	-0.48L
6. Better Late	+1.40L	40	-0.35L	-0.51L	-0.52L	-0.10L
7. Brown Sugar	-0.77L	55	-0.16L	-0.29L	-0.35L	+0.01L
9. Angara	+2.96L	36	-0.20L	-0.27L	+0.04L	+0.34L
10. Miss Checkoni	+3.78L	26	-0.98L	-1.03L	-0.70L	-0.07L
11. Lika Remi	+1.98L	34	-0.74L	-0.46L	-0.24L	+0.07L
12. Incantress	+2.85L	34	-0.65L	-0.62L	-0.27L	+0.14L
"""

df_last_10_runs = pd.read_csv(io.StringIO(last_10_runs_data), sep='\t')

# Clean up 'Runner Name' to extract just the number and rename it 'Tab'
df_last_10_runs['Tab'] = df_last_10_runs['Runner Name'].str.extract('^(\d+)\.').astype(int)
df_last_10_runs['Horse'] = df_last_10_runs['Runner Name'].str.replace('^\d+\. ', '', regex=True)
df_last_10_runs = df_last_10_runs.drop(columns=['Runner Name'])

# Convert 'L' (lengths) to numeric, stripping the 'L' and handling signs
for col in ['Runner Time', 'L800m', 'L600m', 'L400m', 'L200m']:
    df_last_10_runs[col] = df_last_10_runs[col].str.replace('L', '', regex=False).astype(float)

display(df_last_10_runs)

<>:20: SyntaxWarning: invalid escape sequence '\d'
<>:21: SyntaxWarning: invalid escape sequence '\d'
<>:20: SyntaxWarning: invalid escape sequence '\d'
<>:21: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_1127/1818602299.py:20: SyntaxWarning: invalid escape sequence '\d'
  df_last_10_runs['Tab'] = df_last_10_runs['Runner Name'].str.extract('^(\d+)\.').astype(int)
/tmp/ipykernel_1127/1818602299.py:21: SyntaxWarning: invalid escape sequence '\d'
  df_last_10_runs['Horse'] = df_last_10_runs['Runner Name'].str.replace('^\d+\. ', '', regex=True)


,Runner Time,Early Pace,L800m,L600m,L400m,L200m,Tab,Horse
0,-1.69,40,-1.22,-1.16,-0.77,-0.19,1,Cryptonic
1,-0.51,48,-0.68,-0.49,-0.18,0.33,2,Heuristic
2,2.22,31,-1.18,-1.16,-0.93,-0.48,4,Grinzinger Pod
3,1.40,40,-0.35,-0.51,-0.52,-0.10,6,Better Late
4,-0.77,55,-0.16,-0.29,-0.35,0.01,7,Brown Sugar
5,2.96,36,-0.20,-0.27,0.04,0.34,9,Angara
6,3.78,26,-0.98,-1.03,-0.70,-0.07,10,Miss Checkoni
7,1.98,34,-0.74,-0.46,-0.24,0.07,11,Lika Remi
8,2.85,34,-0.65,-0.62,-0.27,0.14,12,Incantress


In [34]:
# Define feature columns for 'LAST 10 RUNS' data
feature_cols_last_10 = ['Runner Time', 'Early Pace', 'L800m', 'L600m', 'L400m', 'L200m']

# Define directional map: 'Early Pace' is higher is better, all others are lower is better
directional_map_last_10 = {
    'Runner Time': 'lower',
    'Early Pace': 'higher',
    'L800m': 'lower',
    'L600m': 'lower',
    'L400m': 'lower',
    'L200m': 'lower'
}

# Detect good outliers using Version 2 of the function
results_last_10_runs = detect_good_outliers_only(
    df_last_10_runs,
    feature_cols=feature_cols_last_10,
    directional_map=directional_map_last_10
)

print("\nGood Outliers for LAST 10 RUNS:")
display(results_last_10_runs[results_last_10_runs['Is_Good_Outlier']])


Good Outliers for LAST 10 RUNS:


,Runner Time,Early Pace,L800m,L600m,L400m,L200m,Tab,Horse,Good_Outlier_Metrics,Max_Good_Z,Is_Good_Outlier
0,-1.69,40,-1.22,-1.16,-0.77,-0.19,1,Cryptonic,"[Runner Time, L800m, L600m]",3.41,True
1,-0.51,48,-0.68,-0.49,-0.18,0.33,2,Heuristic,[Early Pace],3.30,True
2,2.22,31,-1.18,-1.16,-0.93,-0.48,4,Grinzinger Pod,"[L800m, L600m, L400m, L200m]",4.83,True
4,-0.77,55,-0.16,-0.29,-0.35,0.01,7,Brown Sugar,"[Runner Time, Early Pace]",5.38,True
